# Caricamento dataset


L'obiettivo dell'esercitazione è svolgere task di classificazione binaria sulla feature *death* e una classificazione multiclasse sulla feature *dzgroup* e una regressione sulla feature *aps* 

In [7]:
import pandas as pd

df = pd.read_csv("dataset_esercitazione.csv")
print(df.head())
print(df.shape)

        age     sex            dzgroup             dzclass  num.co   edu  \
0  62.84998    male        Lung Cancer              Cancer       0  11.0   
1  60.33899  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
2  52.74698  female          Cirrhosis  COPD/CHF/Cirrhosis       2  12.0   
3  42.38498  female        Lung Cancer              Cancer       2  11.0   
4  79.88495  female  ARF/MOSF w/Sepsis            ARF/MOSF       1   NaN   

       income  scoma  charges  totcst  ...      crea    sod        ph  \
0    $11-$25k    0.0   9715.0     NaN  ...  1.199951  141.0  7.459961   
1    $11-$25k   44.0  34496.0     NaN  ...  5.500000  132.0  7.250000   
2  under $11k    0.0  41094.0     NaN  ...  2.000000  134.0  7.459961   
3  under $11k    0.0   3075.0     NaN  ...  0.799927  139.0       NaN   
4         NaN   26.0  50127.0     NaN  ...  0.799927  143.0  7.509766   

   glucose  bun  urine  adlp  adls  adlsc  death  
0      NaN  NaN    NaN   7.0   7.0    7.0      0  
1 

# Analisi valori NaN e imputazione dati mancanti

In [ ]:
# Sceglieremo una soglia percentuale e andremo ad eliminare le colonne eccessivamente sparse

missing_pct = df.isna().mean(axis=0) * 100
cols_to_drop = missing_pct[missing_pct > 30].index.to_list()

print(missing_pct.sort_values(ascending=False)) # osserviamo i valori mancanti per ciascun colonna

#eliminiamo le feature con percentuale di valori mancanti maggiore del 30%.
df = df.drop(columns=cols_to_drop)
print(f"Colonne eliminate: {cols_to_drop}")

adlp        61.954970
urine       53.399231
glucose     49.423394
bun         47.797913
totmcst     38.165843
alb         37.034596
income      32.751236
adls        31.488193
bili        28.566722
pafi        25.535420
ph          25.085118
prg2m       18.110928
edu         17.946183
prg6m       17.935200
totcst       9.752883
wblc         2.328391
charges      1.889072
avtisst      0.900604
crea         0.735859
race         0.461285
dnr          0.329489
dnrday       0.329489
scoma        0.010983
sod          0.010983
sps          0.010983
meanbp       0.010983
surv2m       0.010983
hrt          0.010983
resp         0.010983
temp         0.010983
aps          0.010983
surv6m       0.010983
adlsc        0.000000
age          0.000000
sex          0.000000
ca           0.000000
dementia     0.000000
diabetes     0.000000
hday         0.000000
num.co       0.000000
dzclass      0.000000
dzgroup      0.000000
death        0.000000
dtype: float64
Colonne eliminate: ['income', 'totmcst'

In [3]:
# Calcoliamo la percentuale di valori mancanti anche per le righe
missing_rows_pct = df.isna().mean(axis=1) * 100
print(missing_rows_pct.describe())

count    9105.000000
mean        4.286185
std         4.170371
min         0.000000
25%         0.000000
50%         2.857143
75%         5.714286
max        42.857143
dtype: float64


In [ ]:
rows_to_drop = missing_rows_pct[missing_rows_pct > 40].index
df = df.drop(index = rows_to_drop)

print(f"Righe eliminate: {len(rows_to_drop)}")


KeyError: '[5393] not found in axis'

In [6]:
print(f"Nuova dimensione: {df.shape}")

Nuova dimensione: (9104, 35)


# Split e preprocessing
Dovendo effettuare tre task differenti andremo ad estrarre tre etichette e creeremo tre DataFrame differenti per ciascun task:

In [ ]:
y_death = df["death"]
y_dzgroup = df["dzgroup"]
y_aps = df["aps"]

X_death = df.drop(columns=["death"])
X_dzgroup = df.drop(columns=["dzgroup"])
X_aps = df.drop(columns=["aps"])



from sklearn.model_selection import train_test_split

X_train_death, X_test_death, y_train_death, y_test_death = train_test_split(X_death, y_death, stratify=y_death, random_state=42, test_size=0.15)
X_train_dzgroup, X_test_dzgroup, y_train_dzgroup, y_test_dzgroup = train_test_split(X_dzgroup, y_dzgroup, stratify=y_dzgroup, random_state=42, test_size=0.15)
X_train_aps, X_test_aps, y_train_aps, y_test_aps = train_test_split(X_aps, y_aps, random_state=42, test_size=0.15)

# PREPROCESSING DEATH
cat_cols = X_train_death.select_dtypes(include=["object", "bool", "category"]).columns
num_cols = X_train_death.columns.drop(cat_cols)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Creiamo la pipeline di preprocessing

# Pipeline per le colonne categoriche
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(fill_value="Unknown", strategy="constant")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Pipeline per le colonne numeriche
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('cat', cat_pipe, cat_cols),
    ('num', num_pipe, num_cols)
])

preprocessor.set_output(transform="pandas")
